# release16_state_aux_loss_training_capability_v1

## Experiment Governance

- **Canonical baseline**: `soxlbacktest_qlib_walkforward_baseline_v1`
- **Direct comparison baseline**: `release16_holdout_slice_repair_candidate_screen_v1`
- **Role**: `diagnose` / training-capability handoff
- **Target window**: `wf_2020`, `wf_2022`, `wf_2023_2024` must later pass `+30pp` relative return and `<=20%` MaxDD before any checkpoint or actor retention.
- **Changed layer**: `bc_warm_start`, `model`, `selection`
- **External reference**: prior Release16 holdout-slice repair cards requiring `drawdown_state_aux_loss` and `release_state_aux_loss`.
- **Hypothesis**: current BC path failed sparse holdout slices because action CE/sample weights only audit state slices after training; adding shared-backbone auxiliary state losses gives the next holdout screen a real training objective for release/downtrend state recognition.
- **Acceptance criteria for this card**: the repo exposes a tested aux-loss BC training contract and a minimal executable proof records high auxiliary state-head fit. This card does not run official replay and cannot promote a model.
- **Closest historical success**: `release16_downtrend_drawdown_objective_screen_v1` fixed MaxDD below 20% but still missed relative/slice gates.
- **Closest historical failure**: `release16_joint_drawdown_aux_state_head_bc_holdout_replay_v1` and `release16_action_head_position_loss_screen_v1` left sparse holdout slices failed.
- **Why this is not a duplicate**: this run adds the missing shared training capability in `src/soxl_quant/app/state_objective_bc.py` and records the next repeat-holdout-screen contract; it does not re-rank old CSVs.

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCAuxLossConfig,
    StateObjectiveBCTrainingConfig,
    fit_bc_actor,
)

ROOT = Path.cwd()
LABEL = "release16_state_aux_loss_training_capability_v1"
CARD_DIR = ROOT / "runtime" / "model_selection" / "card" / LABEL
SOURCE_DIR = ROOT / "runtime" / "model_selection" / "card" / "release16_holdout_slice_repair_candidate_screen_v1"
SOURCE_SUMMARY = json.loads((SOURCE_DIR / "summary.json").read_text(encoding="utf-8"))
CARD_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_SUMMARY["decision"], SOURCE_SUMMARY["required_repair_terms"]

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


('state_objective_holdout_repair_not_ready_no_training',
 ['checkpoint_objective_loss',
  'drawdown_state_aux_loss',
  'release_state_aux_loss',
  'trend_preservation_floor'])

In [2]:
class TinySharedActor(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.preprocess = torch.nn.Sequential(torch.nn.Linear(2, 4), torch.nn.Tanh())
        self.last = torch.nn.Linear(4, 2)

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.last(self.preprocess(obs))


torch.manual_seed(3)
obs = np.asarray(
    [[-2.0, 0.0], [-1.5, 0.0], [-1.0, 0.0], [1.0, 0.0], [1.5, 0.0], [2.0, 0.0]],
    dtype=np.float32,
)
actions = np.asarray([0, 0, 0, 1, 1, 1], dtype=np.int64)
metadata = pd.DataFrame(
    {
        "release_gate_state": [False, False, False, True, True, True],
        "is_downtrend_state": [True, True, False, False, False, False],
    }
)
actor = TinySharedActor()
metrics = fit_bc_actor(
    actor,
    obs,
    actions,
    StateObjectiveBCTrainingConfig(steps=160, lr=0.03, batch_size=6, seed=19),
    aux_state_metadata=metadata,
    aux_loss_config=StateObjectiveBCAuxLossConfig(
        state_columns=("release_gate_state", "is_downtrend_state"),
        loss_weight=1.0,
    ),
)
assert metrics["aux_release_gate_state_after_accuracy"] >= 0.95
assert metrics["aux_is_downtrend_state_after_accuracy"] >= 0.95
assert metrics["after_accuracy"] >= 0.95
metrics

{'samples': 6,
 'steps': 160,
 'lr': 0.03,
 'batch_size': 6,
 'before_loss': 0.7267048358917236,
 'before_accuracy': 0.5,
 'train_last_loss': 0.006477000657469034,
 'train_last_batch_accuracy': 1.0,
 'after_loss': 0.003936745226383209,
 'after_accuracy': 1.0,
 'aux_state_loss_weight': 1.0,
 'aux_state_head_count': 2,
 'aux_state_last_loss': 0.10173605382442474,
 'aux_release_gate_state_after_accuracy': 1.0,
 'aux_release_gate_state_after_loss': 0.01567953700820605,
 'aux_release_gate_state_positive_samples': 3,
 'aux_release_gate_state_before_accuracy': 0.5,
 'aux_release_gate_state_before_loss': 0.8156762917836508,
 'aux_is_downtrend_state_after_accuracy': 1.0,
 'aux_is_downtrend_state_after_loss': 0.10888266563415527,
 'aux_is_downtrend_state_positive_samples': 2,
 'aux_is_downtrend_state_before_accuracy': 0.0,
 'aux_is_downtrend_state_before_loss': 1.041295051574707}

In [3]:
required_terms = list(SOURCE_SUMMARY.get("required_repair_terms", []))
implemented_terms = ["drawdown_state_aux_loss", "release_state_aux_loss"]
remaining_terms = [term for term in required_terms if term not in implemented_terms]
created_at = datetime.now(timezone.utc).isoformat()
training_contract = {
    "experiment": LABEL,
    "created_at": created_at,
    "source_experiment": "release16_holdout_slice_repair_candidate_screen_v1",
    "canonical_baseline": "soxlbacktest_qlib_walkforward_baseline_v1",
    "direct_comparison_baseline": "release16_holdout_slice_repair_candidate_screen_v1",
    "role": "diagnose",
    "changed_layer": ["bc_warm_start", "model", "selection"],
    "aux_state_loss_config": {
        "state_columns": ["release_gate_state", "is_downtrend_state"],
        "loss_weight": 1.0,
    },
    "required_repair_terms": required_terms,
    "implemented_repair_terms": implemented_terms,
    "remaining_repair_terms": remaining_terms,
    "repeat_holdout_screen_required": True,
    "official_replay_required_before_promotion": True,
}
summary = {
    **training_contract,
    "decision": "state_aux_loss_training_capability_ready_repeat_holdout_slice_screen_no_promotion",
    "primary_blocker": "training_contract_ready",
    "target_contract_gate": True,
    "bc_readiness_gate": True,
    "bc_screen_allowed_next": True,
    "bc_aux_loss_screen_allowed_next": True,
    "checkpoint_screen_allowed_next": False,
    "ppo_allowed_next": False,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "target_relative_return_pp": float(SOURCE_SUMMARY.get("target_relative_return_pp", 30.0)),
    "target_max_dd_pct": float(SOURCE_SUMMARY.get("target_max_dd_pct", 20.0)),
    "required_wf_windows": SOURCE_SUMMARY.get("required_wf_windows", ["wf_2020", "wf_2022", "wf_2023_2024"]),
    "source_decision": SOURCE_SUMMARY.get("decision"),
    "source_primary_blocker": SOURCE_SUMMARY.get("primary_blocker"),
    "synthetic_action_after_accuracy": float(metrics["after_accuracy"]),
    "synthetic_release_gate_after_accuracy": float(metrics["aux_release_gate_state_after_accuracy"]),
    "synthetic_downtrend_after_accuracy": float(metrics["aux_is_downtrend_state_after_accuracy"]),
    "next_mechanism": "run_aux_state_loss_bc_holdout_slice_screen_with_trend_preservation_floor_before_checkpoint_screen",
    "sidecars": [
        f"{LABEL}.ipynb",
        "training_contract.json",
        "summary.json",
        "conclusion.md",
    ],
}
(CARD_DIR / "training_contract.json").write_text(json.dumps(training_contract, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(CARD_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
conclusion = "\n".join([
    "# Release16 State Aux Loss Training Capability",
    "",
    f"- decision: `{summary['decision']}`",
    f"- implemented_repair_terms: `{', '.join(implemented_terms)}`",
    f"- remaining_repair_terms: `{', '.join(remaining_terms)}`",
    f"- synthetic_action_after_accuracy: `{summary['synthetic_action_after_accuracy']:.4f}`",
    f"- synthetic_release_gate_after_accuracy: `{summary['synthetic_release_gate_after_accuracy']:.4f}`",
    f"- synthetic_downtrend_after_accuracy: `{summary['synthetic_downtrend_after_accuracy']:.4f}`",
    "- promotion_allowed: `False`",
    "- next_mechanism: `run_aux_state_loss_bc_holdout_slice_screen_with_trend_preservation_floor_before_checkpoint_screen`",
])
(CARD_DIR / "conclusion.md").write_text(conclusion + "\n", encoding="utf-8")
summary

{'experiment': 'release16_state_aux_loss_training_capability_v1',
 'created_at': '2026-06-15T10:45:12.275827+00:00',
 'source_experiment': 'release16_holdout_slice_repair_candidate_screen_v1',
 'canonical_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
 'direct_comparison_baseline': 'release16_holdout_slice_repair_candidate_screen_v1',
 'role': 'diagnose',
 'changed_layer': ['bc_warm_start', 'model', 'selection'],
 'aux_state_loss_config': {'state_columns': ['release_gate_state',
   'is_downtrend_state'],
  'loss_weight': 1.0},
 'required_repair_terms': ['checkpoint_objective_loss',
  'drawdown_state_aux_loss',
  'release_state_aux_loss',
  'trend_preservation_floor'],
 'implemented_repair_terms': ['drawdown_state_aux_loss',
  'release_state_aux_loss'],
 'remaining_repair_terms': ['checkpoint_objective_loss',
  'trend_preservation_floor'],
 'repeat_holdout_screen_required': True,
 'official_replay_required_before_promotion': True,
 'decision': 'state_aux_loss_training_capabilit

## Conclusion

This card turns the prior holdout-slice blocker into an executable training-capability handoff. `fit_bc_actor()` now supports shared-backbone auxiliary state losses, and the proof run trains both `release_gate_state` and `is_downtrend_state` heads to high accuracy on a minimal separable fixture while preserving action-label accuracy.

This is not retention evidence. No official walk-forward replay was run, and no actor is retained. The next valid step is to rerun the Release16 BC holdout-slice screen with aux state loss plus a trend-preservation floor, then only proceed to checkpoint/reward screening if all required folds satisfy the path and sparse-slice gates.